# pandas Part 2 : point-in-time data , ML integration , real-world sources
    1. merge_asof        ---> "what was true AT THAT MOMENT ?"  (point-in-time joins)
    2. encoding for ML   ---> get_dummies pitfalls , OneHotEncoder , ColumnTransformer , set_output
    3. advanced windows  ---> ewm , expanding , time-based rolling , per-group rolling
    4. honest splits     ---> stratified split , group split (same entity in train AND test)
    5. nested JSON       ---> json_normalize , explode
    6. SQL and Excel     ---> sqlite3 + read_sql , multi-sheet Excel
    7. user behaviour    ---> cumsum / cumcount / rank / diff , sessions , funnels
    8. calendars         ---> time zones , Iranian business days , Jalali dates
    9. MultiIndex        ---> xs , IndexSlice , swaplevel
   10. pipelines         ---> pipe , small validation checks
   11. final application ---> a leak-free churn dataset from raw tables + model

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)

pandas 3.0.5


## 1. merge_asof : join on the most recent row BEFORE each timestamp
    an ordinary merge needs EQUAL keys ; time stamps almost never match exactly

In [2]:
plan_history = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3],
    "effective": pd.to_datetime(["2026-01-01", "2026-04-10", "2026-01-15", "2026-06-01", "2026-02-01"]),
    "plan": ["basic", "pro", "pro", "basic", "premium"],
})
tx = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 1],
    "ts": pd.to_datetime(["2026-02-03", "2026-05-20", "2026-03-01", "2026-07-15", "2026-03-09", "2026-04-10"]),
    "amount": [120, 480, 450, 210, 900, 470],
}).sort_values("ts")

current = plan_history.sort_values("effective").groupby("customer_id").tail(1)       # the plan TODAY
naive = tx.merge(current[["customer_id", "plan"]].rename(columns={"plan": "plan_today"}), on="customer_id")

asof = pd.merge_asof(naive.sort_values("ts"), plan_history.sort_values("effective"),
                     left_on="ts", right_on="effective", by="customer_id", direction="backward")
print(asof.rename(columns={"plan": "plan_at_tx_time"}).sort_values(["customer_id", "ts"]))

   customer_id         ts  amount plan_today  effective plan_at_tx_time
0            1 2026-02-03     120        pro 2026-01-01           basic
3            1 2026-04-10     470        pro 2026-04-10             pro
4            1 2026-05-20     480        pro 2026-04-10             pro
1            2 2026-03-01     450      basic 2026-01-15             pro
5            2 2026-07-15     210      basic 2026-06-01           basic
2            3 2026-03-09     900    premium 2026-02-01         premium


In [3]:
# tolerance : do not use a value that is too old (stale exchange rate)
rates = pd.DataFrame({
    "ts": pd.to_datetime(["2026-11-07 09:00", "2026-11-07 10:00", "2026-11-07 11:00"]),
    "usd_rate": [101_500, 101_900, 102_300],
})
trades = pd.DataFrame({
    "ts": pd.to_datetime(["2026-11-07 08:30", "2026-11-07 09:10", "2026-11-07 10:45", "2026-11-07 11:05"]),
    "usd": [100, 250, 40, 75],
})
priced = pd.merge_asof(trades, rates, on="ts", tolerance=pd.Timedelta("30min"))
priced["toman"] = priced["usd"] * priced["usd_rate"]
print(priced)

                   ts  usd  usd_rate       toman
0 2026-11-07 08:30:00  100       NaN         NaN
1 2026-11-07 09:10:00  250  101500.0  25375000.0
2 2026-11-07 10:45:00   40       NaN         NaN
3 2026-11-07 11:05:00   75  102300.0   7672500.0


## 2. Categorical features for ML
    get_dummies is fine for EXPLORATION , dangerous inside a train / test workflow

In [4]:
train = pd.DataFrame({"city": ["Tehran", "Shiraz", "Tehran", "Tabriz"], "age": [30, 41, 25, 38]})
test = pd.DataFrame({"city": ["Shiraz", "Mashhad"], "age": [29, 50]})

d_train = pd.get_dummies(train, columns=["city"], dtype=int)
d_test = pd.get_dummies(test, columns=["city"], dtype=int)
print("train columns:", list(d_train.columns))
print("test  columns:", list(d_test.columns), "<- different shape and order !")

aligned = d_test.reindex(columns=d_train.columns, fill_value=0)       # manual fix
print(aligned)

train columns: ['age', 'city_Shiraz', 'city_Tabriz', 'city_Tehran']
test  columns: ['age', 'city_Mashhad', 'city_Shiraz'] <- different shape and order !
   age  city_Shiraz  city_Tabriz  city_Tehran
0   29            1            0            0
1   50            0            0            0


In [5]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
enc.fit(train[["city"]])                                       # categories learned from TRAIN only
print(enc.categories_)
print(enc.transform(test[["city"]]))                           # unseen 'Mashhad' -> all zeros

[array(['Shiraz', 'Tabriz', 'Tehran'], dtype=object)]
   city_Shiraz  city_Tabriz  city_Tehran
0          1.0          0.0          0.0
1          0.0          0.0          0.0


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

people = pd.DataFrame({
    "age": [34, np.nan, 45, 31, 39],
    "income": [52.0, 61.5, np.nan, 48.0, 70.0],
    "city": ["Tehran", "Shiraz", "Tehran", np.nan, "Tabriz"],
    "plan": ["basic", "pro", "basic", "premium", "pro"],
})
num_cols, cat_cols = ["age", "income"], ["city", "plan"]

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat_cols),
]).set_output(transform="pandas")

X = prep.fit_transform(people)
print(X.round(2))
print("features:", list(X.columns))

   num__age  num__income  cat__city_Shiraz  cat__city_Tabriz  cat__city_Tehran  cat__plan_basic  cat__plan_premium  \
0     -0.65        -0.74               0.0               0.0               1.0              1.0                0.0   
1     -0.13         0.50               1.0               0.0               0.0              0.0                0.0   
2      1.66        -0.12               0.0               0.0               1.0              1.0                0.0   
3     -1.28        -1.26               0.0               0.0               1.0              0.0                1.0   
4      0.40         1.61               0.0               1.0               0.0              0.0                0.0   

   cat__plan_pro  
0            0.0  
1            1.0  
2            0.0  
3            0.0  
4            1.0  
features: ['num__age', 'num__income', 'cat__city_Shiraz', 'cat__city_Tabriz', 'cat__city_Tehran', 'cat__plan_basic', 'cat__plan_premium', 'cat__plan_pro']


## 3. Advanced windows
    rolling(n)        ---> last n rows , equal weights
    ewm(span=n)       ---> ALL past rows , weights decay exponentially (recent = more important)
    expanding()       ---> everything so far
    rolling("7D")     ---> a TIME window : works with irregular timestamps

In [7]:
rng = np.random.default_rng(0)
level = np.r_[np.full(40, 10.0), np.full(40, 20.0)]                 # a sudden change at t = 40
s = pd.Series(level + rng.normal(0, 0.5, 80))

comp = pd.DataFrame({
    "value": s,
    "rolling_10": s.rolling(10).mean(),
    "ewm_span10": s.ewm(span=10, adjust=False).mean(),
    "expanding": s.expanding().mean(),
})
print(comp.iloc[[38, 40, 42, 45, 50, 60, 79]].round(2))

    value  rolling_10  ewm_span10  expanding
38  10.39       10.00       10.03       9.95
40  19.37       11.05       11.83      10.20
42  20.67       13.21       14.77      10.69
45  19.84       16.19       17.17      11.30
50  20.18       20.52       19.36      12.22
60  19.78       19.87       19.78      13.48
79  21.00       20.20       20.23      15.06


In [8]:
# weights : EWM with span = 10  ->  alpha = 2 / (span + 1)
alpha = 2 / (10 + 1)
w = alpha * (1 - alpha) ** np.arange(8)
print("alpha =", round(alpha, 3), "| weights of the last 8 points:", w.round(3))
print("halflife =", round(np.log(0.5) / np.log(1 - alpha), 2), "steps")

alpha = 0.182 | weights of the last 8 points: [0.182 0.149 0.122 0.1   0.081 0.067 0.055 0.045]
halflife = 3.45 steps


In [9]:
# time-based windows on IRREGULAR events : transactions per card in the previous hour (fraud velocity)
cards = pd.DataFrame({
    "card": ["C1", "C1", "C1", "C1", "C2", "C2", "C1"],
    "ts": pd.to_datetime(["2026-11-07 10:00", "2026-11-07 10:20", "2026-11-07 10:25", "2026-11-07 10:31",
                          "2026-11-07 10:05", "2026-11-07 13:00", "2026-11-07 12:00"]),
    "amount": [50, 900, 950, 990, 30, 40, 60],
}).sort_values(["card", "ts"], ignore_index=True)

roll = (cards.set_index("ts")
             .groupby("card")["amount"]
             .rolling("1h", closed="left")                       # closed="left" : exclude the current transaction
             .agg(["count", "sum"]))
cards[["tx_last_hour", "amount_last_hour"]] = roll.to_numpy()     # same order : sorted by card , ts
cards["avg_before"] = cards.groupby("card")["amount"].transform(lambda a: a.shift(1).expanding().mean())
cards[["tx_last_hour", "amount_last_hour"]] = cards[["tx_last_hour", "amount_last_hour"]].fillna(0)
print(cards.round({"avg_before": 1}))

  card                  ts  amount  tx_last_hour  amount_last_hour  avg_before
0   C1 2026-11-07 10:00:00      50           0.0               0.0         NaN
1   C1 2026-11-07 10:20:00     900           1.0              50.0        50.0
2   C1 2026-11-07 10:25:00     950           2.0             950.0       475.0
3   C1 2026-11-07 10:31:00     990           3.0            1900.0       633.3
4   C1 2026-11-07 12:00:00      60           0.0               0.0       722.5
5   C2 2026-11-07 10:05:00      30           0.0               0.0         NaN
6   C2 2026-11-07 13:00:00      40           0.0               0.0        30.0


## 4. Honest train / test splits

In [10]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(1)
y = pd.Series((rng.random(1000) < 0.05).astype(int))                # rare positive class : 5 %
rows = []
for seed in range(5):
    _, yt = train_test_split(y, test_size=100, random_state=seed)
    _, ys = train_test_split(y, test_size=100, random_state=seed, stratify=y)
    rows.append({"seed": seed, "random_split_pos_rate": yt.mean(), "stratified_pos_rate": ys.mean()})
print("full data positive rate:", y.mean())
print(pd.DataFrame(rows))

full data positive rate: 0.047
   seed  random_split_pos_rate  stratified_pos_rate
0     0                   0.08                 0.05
1     1                   0.01                 0.05
2     2                   0.01                 0.05
3     3                   0.06                 0.05
4     4                   0.03                 0.05


In [11]:
# GROUP leakage : 200 patients , 5 visits each ; the label belongs to the PATIENT and is pure noise
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(2)
n_pat, visits = 200, 5
signature = rng.normal(size=(n_pat, 6))                               # each patient is "recognisable"
pid = np.repeat(np.arange(n_pat), visits)
X = signature[pid] + rng.normal(0, 0.1, (n_pat * visits, 6))
y = rng.integers(0, 2, n_pat)[pid]                                     # no real signal at all

def score(train_idx, test_idx):
    model = KNeighborsClassifier(n_neighbors=1).fit(X[train_idx], y[train_idx])
    return model.score(X[test_idx], y[test_idx])

idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.25, random_state=0)
print("random split : patients in both sets =", len(set(pid[tr]) & set(pid[te])), "| accuracy =", round(score(tr, te), 3))

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(X, y, groups=pid))
print("group split  : patients in both sets =", len(set(pid[tr]) & set(pid[te])), "| accuracy =", round(score(tr, te), 3))

random split : patients in both sets = 153 | accuracy = 1.0
group split  : patients in both sets = 0 | accuracy = 0.452


## 5. Nested JSON from an API : json_normalize and explode

In [12]:
api_response = [
    {"order_id": 501, "customer": {"id": 7, "city": "Tehran", "vip": True},
     "items": [{"sku": "A1", "qty": 2, "price": 120}, {"sku": "B4", "qty": 1, "price": 900}],
     "tags": ["express", "gift"]},
    {"order_id": 502, "customer": {"id": 9, "city": "Shiraz"},
     "items": [{"sku": "A1", "qty": 1, "price": 120}],
     "tags": []},
]

orders = pd.json_normalize(api_response, sep="_")                       # nested dicts -> columns
print(orders[["order_id", "customer_id", "customer_city", "customer_vip", "tags"]])

items = pd.json_normalize(api_response, record_path="items",            # one row per ITEM
                          meta=["order_id", ["customer", "id"]], sep="_")
items["line_total"] = items["qty"] * items["price"]
print(items)
print(items.groupby("order_id")["line_total"].sum())

print(orders[["order_id", "tags"]].explode("tags"))                     # list in a cell -> rows

   order_id  customer_id customer_city customer_vip             tags
0       501            7        Tehran         True  [express, gift]
1       502            9        Shiraz          NaN               []
  sku  qty  price order_id customer_id  line_total
0  A1    2    120      501           7         240
1  B4    1    900      501           7         900
2  A1    1    120      502           9         120
order_id
501    1140
502     120
Name: line_total, dtype: int64
   order_id     tags
0       501  express
0       501     gift
1       502      NaN


## 6. SQL databases and Excel workbooks

In [13]:
import sqlite3

con = sqlite3.connect(":memory:")                                       # a real database , in memory
items.to_sql("order_items", con, index=False)
orders[["order_id", "customer_id", "customer_city"]].to_sql("orders", con, index=False)

query = '''
SELECT o.customer_city AS city , SUM(i.qty * i.price) AS revenue , COUNT(*) AS lines
FROM order_items i JOIN orders o ON i.order_id = o.order_id
WHERE i.price >= ?
GROUP BY o.customer_city
ORDER BY revenue DESC
'''
print(pd.read_sql(query, con, params=(100,)))
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con))
con.close()

     city  revenue  lines
0  Tehran     1140      2
1  Shiraz      120      1
          name
0  order_items
1       orders


In [14]:
from pathlib import Path
out = Path("pandas2_demo")
out.mkdir(exist_ok=True)

with pd.ExcelWriter(out / "report.xlsx") as writer:
    orders.drop(columns=["items", "tags"]).to_excel(writer, sheet_name="orders", index=False)
    items.to_excel(writer, sheet_name="items", index=False)

sheets = pd.read_excel(out / "report.xlsx", sheet_name=None)             # None -> ALL sheets as a dict
for name, frame in sheets.items():
    print(name, frame.shape)
print(pd.read_excel(out / "report.xlsx", sheet_name="items", usecols=["sku", "qty"]))

orders (2, 4)
items (3, 6)
  sku  qty
0  A1    2
1  B4    1
2  A1    1


## 7. User behaviour : cumulative tools , sessions , funnels

In [15]:
clicks = pd.DataFrame({
    "user": ["u1"] * 7 + ["u2"] * 5,
    "ts": pd.to_datetime([
        "2026-11-07 09:00", "2026-11-07 09:05", "2026-11-07 09:20", "2026-11-07 11:00",
        "2026-11-07 11:10", "2026-11-07 11:12", "2026-11-07 20:00",
        "2026-11-07 10:00", "2026-11-07 10:40", "2026-11-07 10:45", "2026-11-07 10:50", "2026-11-07 10:52"]),
    "event": ["view", "view", "cart", "view", "cart", "purchase", "view",
              "view", "view", "cart", "view", "purchase"],
}).sort_values(["user", "ts"], ignore_index=True)

gap = clicks.groupby("user")["ts"].diff()
clicks["new_session"] = gap.isna() | (gap > pd.Timedelta("30min"))
clicks["session"] = clicks.groupby("user")["new_session"].cumsum()       # 1 , 1 , 1 , 2 , ...
clicks["step_in_session"] = clicks.groupby(["user", "session"]).cumcount() + 1
print(clicks)

   user                  ts     event  new_session  session  step_in_session
0    u1 2026-11-07 09:00:00      view         True        1                1
1    u1 2026-11-07 09:05:00      view        False        1                2
2    u1 2026-11-07 09:20:00      cart        False        1                3
3    u1 2026-11-07 11:00:00      view         True        2                1
4    u1 2026-11-07 11:10:00      cart        False        2                2
5    u1 2026-11-07 11:12:00  purchase        False        2                3
6    u1 2026-11-07 20:00:00      view         True        3                1
7    u2 2026-11-07 10:00:00      view         True        1                1
8    u2 2026-11-07 10:40:00      view         True        2                1
9    u2 2026-11-07 10:45:00      cart        False        2                2
10   u2 2026-11-07 10:50:00      view        False        2                3
11   u2 2026-11-07 10:52:00  purchase        False        2                4

In [16]:
sessions = clicks.groupby(["user", "session"]).agg(
    start=("ts", "min"), end=("ts", "max"), events=("event", "size"),
    viewed=("event", lambda e: (e == "view").any()),
    carted=("event", lambda e: (e == "cart").any()),
    bought=("event", lambda e: (e == "purchase").any()),
)
sessions["minutes"] = (sessions["end"] - sessions["start"]).dt.total_seconds() / 60
print(sessions)

funnel = sessions[["viewed", "carted", "bought"]].sum()
print(pd.DataFrame({"sessions": funnel, "rate_vs_previous": (funnel / funnel.shift(1)).round(2)}))

                           start                 end  events  viewed  carted  bought  minutes
user session                                                                                 
u1   1       2026-11-07 09:00:00 2026-11-07 09:20:00       3    True    True   False     20.0
     2       2026-11-07 11:00:00 2026-11-07 11:12:00       3    True    True    True     12.0
     3       2026-11-07 20:00:00 2026-11-07 20:00:00       1    True   False   False      0.0
u2   1       2026-11-07 10:00:00 2026-11-07 10:00:00       1    True   False   False      0.0
     2       2026-11-07 10:40:00 2026-11-07 10:52:00       4    True    True    True     12.0
        sessions  rate_vs_previous
viewed         5               NaN
carted         3              0.60
bought         2              0.67


In [17]:
scores = pd.DataFrame({"student": list("ABCDEF"), "score": [18, 20, 18, 15, 20, 12]})
for method in ["average", "min", "dense", "first"]:
    scores[method] = scores["score"].rank(method=method, ascending=False)
scores["percentile"] = scores["score"].rank(pct=True).round(2)
print(scores)

  student  score  average  min  dense  first  percentile
0       A     18      3.5  3.0    2.0    3.0        0.58
1       B     20      1.5  1.0    1.0    1.0        0.92
2       C     18      3.5  3.0    2.0    4.0        0.58
3       D     15      5.0  5.0    3.0    5.0        0.33
4       E     20      1.5  1.0    1.0    2.0        0.92
5       F     12      6.0  6.0    4.0    6.0        0.17


## 8. Time zones , Iranian business days , Jalali dates

In [18]:
server_log = pd.Series(pd.to_datetime(["2026-11-07 05:00", "2026-11-07 21:00"]))
utc = server_log.dt.tz_localize("UTC")                            # "these times ARE in UTC"
tehran = utc.dt.tz_convert("Asia/Tehran")                         # "show them in Tehran time"
print(pd.DataFrame({"utc": utc, "tehran": tehran, "tehran_date": tehran.dt.date}))

try:
    utc.iloc[0] - server_log.iloc[0]
except TypeError as e:
    print("TypeError:", e)

                        utc                    tehran tehran_date
0 2026-11-07 05:00:00+00:00 2026-11-07 08:30:00+03:30  2026-11-07
1 2026-11-07 21:00:00+00:00 2026-11-08 00:30:00+03:30  2026-11-08
TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.


In [19]:
iran_days = pd.bdate_range("2026-11-01", "2026-11-14", freq="C",
                           weekmask="Sat Sun Mon Tue Wed", holidays=["2026-11-09"])     # Mon 9 : an example holiday
print([d.strftime("%a %d") for d in iran_days])

order_day = pd.Timestamp("2026-11-04")                            # Wednesday
due = order_day + pd.offsets.CustomBusinessDay(3, weekmask="Sat Sun Mon Tue Wed", holidays=["2026-11-09"])     # Mon 9 : an example holiday
print("ordered", order_day.strftime("%a %d"), "-> due after 3 working days:", due.strftime("%a %d"))

['Sun 01', 'Mon 02', 'Tue 03', 'Wed 04', 'Sat 07', 'Sun 08', 'Tue 10', 'Wed 11', 'Sat 14']
ordered Wed 04 -> due after 3 working days: Tue 10


In [20]:
def gregorian_to_jalali(gy, gm, gd):
    # arithmetic algorithm (33-year cycle) , valid for modern dates
    g_d_m = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    gy2 = gy + 1 if gm > 2 else gy
    days = 355666 + 365 * gy + (gy2 + 3) // 4 - (gy2 + 99) // 100 + (gy2 + 399) // 400 + gd + g_d_m[gm - 1]
    jy = -1595 + 33 * (days // 12053)
    days %= 12053
    jy += 4 * (days // 1461)
    days %= 1461
    if days > 365:
        jy += (days - 1) // 365
        days = (days - 1) % 365
    if days < 186:
        jm, jd = 1 + days // 31, 1 + days % 31
    else:
        jm, jd = 7 + (days - 186) // 30, 1 + (days - 186) % 30
    return jy, jm, jd

dates = pd.Series(pd.to_datetime(["2026-03-20", "2026-03-21", "2026-09-23", "2026-11-07"]))
jal = dates.map(lambda d: gregorian_to_jalali(d.year, d.month, d.day))
months = ["Farvardin", "Ordibehesht", "Khordad", "Tir", "Mordad", "Shahrivar",
          "Mehr", "Aban", "Azar", "Dey", "Bahman", "Esfand"]
print(pd.DataFrame({
    "gregorian": dates.dt.date,
    "jalali": jal.map(lambda t: f"{t[0]}/{t[1]:02d}/{t[2]:02d}"),
    "month": jal.map(lambda t: months[t[1] - 1]),
}))

    gregorian      jalali      month
0  2026-03-20  1404/12/29     Esfand
1  2026-03-21  1405/01/01  Farvardin
2  2026-09-23  1405/07/01       Mehr
3  2026-11-07  1405/08/16       Aban


## 9. MultiIndex : several index levels

In [21]:
rng = np.random.default_rng(4)
idx = pd.MultiIndex.from_product([["Tehran", "Shiraz"], ["basic", "pro"], [2025, 2026]],
                                 names=["city", "plan", "year"])
rev = pd.DataFrame({"revenue": rng.integers(100, 500, len(idx)), "users": rng.integers(10, 60, len(idx))}, index=idx).sort_index()
print(rev)

print(rev.loc["Tehran"])                                    # first level
print(rev.loc[("Tehran", "pro")])                           # first two levels
print(rev.xs(2026, level="year"))                            # any level by name
print(rev.loc[pd.IndexSlice[:, "pro", 2026], "revenue"])     # slice inside levels
print(rev["revenue"].unstack("year"))                        # level -> columns
print(rev.groupby(level="city").sum())

                   revenue  users
city   plan  year                
Shiraz basic 2025      476     41
             2026      490     50
       pro   2025      488     39
             2026      132     18
Tehran basic 2025      390     32
             2026      477     40
       pro   2025      452     24
             2026      304     28
            revenue  users
plan  year                
basic 2025      390     32
      2026      477     40
pro   2025      452     24
      2026      304     28
      revenue  users
year                
2025      452     24
2026      304     28
              revenue  users
city   plan                 
Shiraz basic      490     50
       pro        132     18
Tehran basic      477     40
       pro        304     28
city    plan  year
Shiraz  pro   2026    132
Tehran  pro   2026    304
Name: revenue, dtype: int64
year          2025  2026
city   plan             
Shiraz basic   476   490
       pro     488   132
Tehran basic   390   477
       pro     4

## 10. Readable pipelines with pipe , plus cheap validation

In [22]:
raw = pd.DataFrame({
    "Customer ID": [1, 2, 2, 3, 4],
    " City ": ["tehran ", "SHIRAZ", "SHIRAZ", None, "Tabriz"],
    "Spend": ["1,200", "800", "800", "۵۰۰", "-"],
})
persian = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

def clean_names(d):
    return d.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))

def drop_dupes(d, key):
    return d.drop_duplicates(subset=key)

def fix_types(d):
    return d.assign(
        city=d["city"].str.strip().str.title(),
        spend=pd.to_numeric(d["spend"].str.translate(persian).str.replace(",", ""), errors="coerce"),
    )

def validate(d):
    assert d["customer_id"].is_unique, "duplicate customer ids"
    assert (d["spend"].dropna() >= 0).all(), "negative spend"
    bad = d["spend"].isna().mean()
    if bad > 0.3:
        raise ValueError(f"too many unparsable spend values: {bad:.0%}")
    return d

tidy = raw.pipe(clean_names).pipe(drop_dupes, key="customer_id").pipe(fix_types).pipe(validate)
print(tidy)
print(tidy.dtypes)

   customer_id    city   spend
0            1  Tehran  1200.0
1            2  Shiraz   800.0
3            3     NaN   500.0
4            4  Tabriz     NaN
customer_id      int64
city               str
spend          float64
dtype: object


## 11. Final application : a leak-free churn dataset from raw tables
    raw tables : customers , plan_history , logins , tickets , cancellations
    for every SNAPSHOT date s :
        features ---> only events BEFORE s
        label    ---> did the customer cancel in (s , s + 30 days] ?

In [23]:
rng = np.random.default_rng(2026)
N = 1500
DATA_END = pd.Timestamp("2026-10-31")
signup = pd.Timestamp("2025-10-01") + pd.to_timedelta(rng.integers(0, 273, N), unit="D")
plan0 = rng.choice(["basic", "pro", "premium"], N, p=[0.6, 0.3, 0.1])
hazard = pd.Series(plan0).map({"basic": 0.09, "pro": 0.045, "premium": 0.03}).to_numpy()     # per month
churn_at = signup + pd.to_timedelta(rng.exponential(30 / hazard), unit="D")

customers = pd.DataFrame({"customer_id": np.arange(N), "signup": signup,
                          "city": rng.choice(["Tehran", "Isfahan", "Shiraz", "Tabriz", "Mashhad"], N)})

upgrade = (plan0 == "basic") & (rng.random(N) < 0.25)
up_time = signup + pd.to_timedelta(rng.integers(10, 200, N), unit="D")
plan_history = pd.concat([
    pd.DataFrame({"customer_id": np.arange(N), "effective": signup, "plan": plan0}),
    pd.DataFrame({"customer_id": np.arange(N)[upgrade], "effective": up_time[upgrade], "plan": "pro"}),
]).sort_values("effective", ignore_index=True)

cancellations = pd.DataFrame({"customer_id": np.arange(N), "cancel_ts": churn_at})
cancellations = cancellations[cancellations["cancel_ts"] <= DATA_END].reset_index(drop=True)

login_rows, ticket_rows = [], []
for c in range(N):
    end = min(churn_at[c], DATA_END)
    span = (end - signup[c]).days
    if span <= 0:
        continue
    lam = rng.lognormal(np.log(0.6), 0.5)                                  # logins per day
    t = rng.uniform(0, span, rng.poisson(lam * span))
    days_left = span - t
    near_end = churn_at[c] <= DATA_END
    keep = rng.random(t.size) < np.where(near_end & (days_left < 30), 0.2 + 0.8 * days_left / 30, 1.0)
    login_rows.append(pd.DataFrame({"customer_id": c, "ts": signup[c] + pd.to_timedelta(t[keep], unit="D")}))
    rate = np.where(near_end & (days_left < 30), 0.08, 0.015)
    tk = t[rng.random(t.size) < rate / lam]
    ticket_rows.append(pd.DataFrame({"customer_id": c, "ts": signup[c] + pd.to_timedelta(tk, unit="D")}))

logins = pd.concat(login_rows, ignore_index=True).sort_values("ts", ignore_index=True)
tickets = pd.concat(ticket_rows, ignore_index=True).sort_values("ts", ignore_index=True)
for name, t in [("customers", customers), ("plan_history", plan_history), ("logins", logins),
                ("tickets", tickets), ("cancellations", cancellations)]:
    print(f"{name:14s} {t.shape}")

customers      (1500, 3)
plan_history   (1710, 3)
logins         (194174, 2)
tickets        (5536, 2)
cancellations  (645, 2)


In [24]:
def window_count(events, s, start_days, end_days, name):
    w = events[(events["ts"] >= s - pd.Timedelta(days=start_days)) & (events["ts"] < s - pd.Timedelta(days=end_days))]
    return w.groupby("customer_id").size().rename(name)

def build_snapshot(s):
    cancel = cancellations.set_index("customer_id")["cancel_ts"]
    active = customers[(customers["signup"] < s - pd.Timedelta(days=14))].copy()
    active = active[~active["customer_id"].map(cancel).le(s).fillna(False).astype(bool)]  # still a customer at s
    active["snapshot"] = s

    past = logins[logins["ts"] < s]
    feats = pd.concat([
        window_count(logins, s, 30, 0, "logins_30d"),
        window_count(logins, s, 60, 30, "logins_prev_30d"),
        window_count(tickets, s, 30, 0, "tickets_30d"),
        past.groupby("customer_id")["ts"].max().rename("last_login"),
    ], axis=1)
    d = active.merge(feats, left_on="customer_id", right_index=True, how="left")
    d[["logins_30d", "logins_prev_30d", "tickets_30d"]] = d[["logins_30d", "logins_prev_30d", "tickets_30d"]].fillna(0)
    d["days_since_login"] = (s - d["last_login"]).dt.days
    d["tenure_days"] = (s - d["signup"]).dt.days
    d["login_trend"] = (d["logins_30d"] + 1) / (d["logins_prev_30d"] + 1)

    d = pd.merge_asof(d.sort_values("snapshot"), plan_history.sort_values("effective"),
                      left_on="snapshot", right_on="effective", by="customer_id", direction="backward")

    c_ts = d["customer_id"].map(cancel)
    d["label"] = ((c_ts > s) & (c_ts <= s + pd.Timedelta(days=30))).astype(int)
    return d.drop(columns=["signup", "last_login", "effective"])

snapshots = pd.date_range("2026-04-01", "2026-09-01", freq="MS")
dataset = pd.concat([build_snapshot(s) for s in snapshots], ignore_index=True)
print(dataset.shape)
print(dataset.groupby("snapshot").agg(customers=("customer_id", "size"), churn_rate=("label", "mean")).round(3))
print(dataset.head())

(5637, 11)
            customers  churn_rate
snapshot                         
2026-04-01        770       0.081
2026-05-01        859       0.083
2026-06-01        939       0.066
2026-07-01       1050       0.070
2026-08-01       1042       0.060
2026-09-01        977       0.054
   customer_id     city   snapshot  logins_30d  logins_prev_30d  tickets_30d  days_since_login  tenure_days  login_trend   plan  \
0            1  Isfahan 2026-04-01        11.0             19.0          0.0               2.0          134     0.600000  basic   
1          970   Shiraz 2026-04-01        17.0              8.0          0.0               0.0          169     2.000000    pro   
2          975   Tabriz 2026-04-01        31.0             18.0          0.0               0.0           47     1.684211  basic   
3          979   Tehran 2026-04-01         9.0             12.0          0.0               0.0          146     0.769231    pro   
4          980   Tehran 2026-04-01        20.0             18.

In [25]:
print(dataset.groupby("label")[["logins_30d", "login_trend", "days_since_login", "tickets_30d", "tenure_days"]]
             .mean().round(2))
print(pd.crosstab(dataset["plan"], dataset["label"], normalize="index").round(3))

       logins_30d  login_trend  days_since_login  tickets_30d  tenure_days
label                                                                     
0           20.42         2.34              1.46         0.43       134.06
1           16.58         1.90              2.88         1.52       125.41
label        0      1
plan                 
basic    0.911  0.089
premium  0.972  0.028
pro      0.943  0.057


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

num = ["logins_30d", "logins_prev_30d", "login_trend", "days_since_login", "tickets_30d", "tenure_days"]
cat = ["plan", "city"]

def make_model():
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])

train = dataset[dataset["snapshot"] < "2026-09-01"]
test = dataset[dataset["snapshot"] == "2026-09-01"]                       # TIME split : the future snapshot
model = make_model().fit(train[num + cat], train["label"])
p = model.predict_proba(test[num + cat])[:, 1]
print("train rows", len(train), "| test rows", len(test), "| test churn rate", round(test["label"].mean(), 3))
print("test AUC:", round(roc_auc_score(test["label"], p), 3))

top = test.assign(risk=p).nlargest(len(test) // 10, "risk")
print("churn rate in top 10 % risk:", round(top["label"].mean(), 3),
      f"-> lift {top['label'].mean() / test['label'].mean():.1f}x")

coefs = pd.Series(model[-1].coef_[0], index=model[0].get_feature_names_out()).sort_values()
print(coefs.round(2))

train rows 4660 | test rows 977 | test churn rate 0.054
test AUC: 0.842
churn rate in top 10 % risk: 0.299 -> lift 5.5x
num__logins_30d         -0.79
cat__plan_premium       -0.58
num__tenure_days        -0.19
cat__city_Mashhad       -0.18
cat__city_Shiraz        -0.13
cat__city_Tehran         0.01
cat__plan_pro            0.08
num__login_trend         0.12
cat__city_Isfahan        0.13
cat__city_Tabriz         0.18
num__days_since_login    0.31
cat__plan_basic          0.50
num__logins_prev_30d     0.61
num__tickets_30d         0.92
dtype: float64


In [27]:
# the SAME model with ONE leaky feature : days since last login measured at DATA_END , not at the snapshot
last_ever = logins.groupby("customer_id")["ts"].max()
leaky = dataset.assign(days_since_login=(DATA_END - dataset["customer_id"].map(last_ever)).dt.days)

tr, te = leaky[leaky["snapshot"] < "2026-09-01"], leaky[leaky["snapshot"] == "2026-09-01"]
m2 = make_model().fit(tr[num + cat], tr["label"])
print("test AUC with the leaky feature:", round(roc_auc_score(te["label"], m2.predict_proba(te[num + cat])[:, 1]), 3))
print("... but at prediction time (today) this feature does not exist for the future -> the model will fail")

CLEANUP = True
if CLEANUP:
    for f in out.iterdir():
        f.unlink()
    out.rmdir()

test AUC with the leaky feature: 0.958
... but at prediction time (today) this feature does not exist for the future -> the model will fail


## Exercises
1. in section 1 , use `direction="forward"` : what does it mean , and when is it a LEAK ?
2. build the `ColumnTransformer` of section 2 with `OneHotEncoder(drop="first")` : how many columns now , and why ?
3. compare `ewm(halflife=3)` with `rolling(6)` after the step change in section 3 : which reacts faster ?
4. in section 3 , add a feature "amount / average amount before" for each card transaction : which one looks like fraud ?
5. in section 4 , use 5-fold `GroupKFold` instead of one group split and report the mean accuracy.
6. change the session gap in section 7 to 10 minutes : how many sessions does each user have now ?
7. write `jalali_month_start(df, col)` that marks rows on the 1st of a Jalali month.
8. in section 11 , remove `days_since_login` and `login_trend` : how much does the AUC drop ?
9. in section 11 , use a 60-day label window instead of 30 : what happens to the churn rate and the AUC ?
10. find another feature in section 11 that could leak if computed carelessly , and explain how.